# Notebook 00 — Python, pandas, and Jupyter Bootcamp

This short bootcamp notebook is a **low-stakes diagnostic** for the start of the NLP course.

It is designed for students who have **some basic Python** but may not be equally confident with:

- notebook mechanics
- variables, lists, and functions
- loading and inspecting tabular data
- filtering and summarising data with pandas
- saving small outputs to disk

We will use a real course file, `./analysis/tables/pg_catalog.csv`, so the practice already connects to the dataset you will encounter later in the semester.

## How this notebook works

For each exercise:

1. run the prompt cell
2. write your answer in the next code cell
3. run the check cell for immediate feedback
4. if needed, ask for a hint with `bc.show_hint("qX")`

This notebook is **not graded**. Its purpose is to help you and the instructor see what already feels comfortable and what may need a quick review.


### During the course, you can use AI assistants freely.

### However, for this notebook you are asked **not to used AI assistance**. The notebook is meant for your instructor to assess your level, not AI assistants.

In [ ]:
!pip install pandas

In [ ]:
from pathlib import Path
import pandas as pd

import nb00_bootcamp_checks as bc
import nb00_bootcamp_solutions as sol

print("Bootcamp helpers loaded.")
print("Catalog path:", bc.CATALOG_PATH)


## A quick Jupyter check

Run the next cell **more than once**. Notice that notebook state is remembered across cells.

Then try this once during class or at home:

- restart the kernel
- rerun the setup cell
- rerun the counter cell

This is a simple reminder that in notebooks, **execution order matters**.


In [ ]:
try:
    bootcamp_counter += 1
except NameError:
    bootcamp_counter = 1

print("bootcamp_counter =", bootcamp_counter)


## Part A — Python warm-up


In [ ]:
bc.ask("q1")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q1(course_name)


In [ ]:
bc.ask("q2")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================

In [ ]:
bc.check_q2(favorite_columns)


In [ ]:
bc.ask("q3")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q3(count_missing)


## Part B — pandas and file I/O with `pg_catalog.csv`


In [ ]:
bc.ask("q4")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q4(catalog_raw)


In [ ]:
bc.ask("q5")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q5(catalog_raw, catalog)


In [ ]:
bc.ask("q6")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q6(catalog, n_rows, n_cols)


In [ ]:
bc.ask("q7")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================



In [ ]:
bc.check_q7(catalog, short_table)


In [ ]:
bc.ask("q8")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q8(catalog, english_books)


In [ ]:
bc.ask("q9")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q9(catalog, top_languages)


In [ ]:
bc.ask("q10")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q10(catalog, recent_catalog)


In [ ]:
bc.ask("q11")


In [ ]:
# =============================================== YOUR CODE HERE ===============================================


In [ ]:
bc.check_q11(sample_path, recent_catalog)


---
When you save a DataFrame to CSV, pandas loses all type information (e.g., Int64 vs int64, string vs str). When you read it back, pandas guesses the types. If your original recent_catalog uses nullable dtypes (like Int64 to support missing values), the saved CSV will load the values with the standard dtypes int64. Although the two numbers look identitcal, in Python, they are different. If you want to compare them, you can compare the *values* or you can save the data types and use them when you load the data that need to be correctly formatted:

```
data_types = df.dtypes.to_dict())

new_df.astype(data_types)
```

Optional: you can run the following cell if you want to understand the issue.

In [ ]:
# =============================================================================
# UNDERSTANDING DTYPES: int64 vs Int64
# =============================================================================
# 
# - int64  (lowercase 'i') = Standard NumPy integer. CANNOT hold NaN/NA.
# - Int64  (capital 'I')   = Pandas nullable integer. CAN hold pd.NA (missing values).
# 
# Even if there are NO missing values, Pandas keeps track of which type you used.
# When you save to CSV and read it back, Pandas forgets the "nullable" part and
# defaults back to standard int64.
# =============================================================================

import pandas as pd
import io  # Used to simulate saving/loading CSV without creating a real file

print("=" * 60)
print("STEP 1: Create an 'Original' DataFrame with nullable Int64 dtypes")
print("=" * 60)

# Imagine this is your 'recent_catalog' – using Int64 (capital I)
original = pd.DataFrame({
    'pg_id': [101, 102, 103, 104],
    'issued_year': [2020, 2021, 2022, 2023]
}).astype({'pg_id': 'Int64', 'issued_year': 'Int64'})

print("Original DataFrame:")
print(original)
print("\nOriginal dtypes:")
print(original.dtypes)
# Output will show:
# pg_id          Int64
# issued_year    Int64


print("\n" + "=" * 60)
print("STEP 2: Simulate saving to CSV and reading it back")
print("=" * 60)

# Convert to CSV string (simulates saving to .csv file)
csv_buffer = io.StringIO()
original.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # Rewind to the beginning so we can read it

# Read it back (simulates a student's pd.read_csv())
loaded = pd.read_csv(csv_buffer)

print("Loaded DataFrame (from CSV):")
print(loaded)
print("\nLoaded dtypes (notice they changed!):")
print(loaded.dtypes)
# Output will show:
# pg_id          int64   <-- lower case! 
# issued_year    int64   <-- lower case!


print("\n" + "=" * 60)
print("STEP 3: The .equals() TRAP")
print("=" * 60)

# Check if the values are the same (they are!)
values_match = loaded.values.tolist() == original.values.tolist()
print(f"Do the values match exactly? {values_match}")  # True

# Check .equals() - which compares dtypes too!
equals_match = loaded.equals(original)
print(f"Does loaded.equals(original) return True? {equals_match}")  # FALSE!
print("Reason: .equals() checks BOTH values AND dtypes. The dtypes differ.")


print("\n" + "=" * 60)
print("STEP 4: WHY DOES Int64 EXIST? (The 'Nullable' Superpower)")
print("=" * 60)

# Create a column with a missing value (None / NA)
nullable_example = pd.DataFrame({
    'standard_int': [1, 2, None],  # Pandas forces this to float64 because int64 can't hold None
    'nullable_int': [1, 2, pd.NA]  # This stays as Int64 and keeps the missing value as NA
})
nullable_example = nullable_example.astype({'nullable_int': 'Int64'})

print("Example with missing values:")
print(nullable_example)
print("\nDtypes:")
print(nullable_example.dtypes)
# Notice:
# standard_int -> float64  (because int64 can't store None, Pandas compromises by using float)
# nullable_int -> Int64    (Pandas keeps it as integer AND supports pd.NA)


print("\n" + "=" * 60)
print("STEP 5: HOW TO FIX THE COMPARISON")
print("=" * 60)

# Option A: Cast the loaded DataFrame to match the original dtypes
loaded_fixed = loaded.astype(original.dtypes.to_dict())
print("Option A - Cast loaded to original dtypes:")
print(f"loaded_fixed.equals(original) -> {loaded_fixed.equals(original)}")  # True

# Option B: Compare while ignoring dtypes (used in the grading checker)
print("\nOption B - Use check_dtype=False in assert_frame_equal:")
try:
    pd.testing.assert_frame_equal(original, loaded, check_dtype=False)
    print("✅ Comparison passed (ignored dtypes). Values are identical.")
except AssertionError:
    print("❌ Values actually differ (rare).")

print("\n" + "=" * 60)
print("Lesson: if you get errors with dataframe, always check the type of data you have.")
print("=" * 60)

## Reflection

If most checks passed quickly, you are probably ready for the course at the expected level.

If some tasks were difficult, that is completely fine. The goal of this notebook is to identify where you may want a short review before the NLP workflow becomes more demanding.

Questions to ask yourself:

- Where did you feel uncomfortable?
- Could you read the feedback and fix errors easily?
- Do notebook state and execution order make sense?
- Would you benefit from a short refresher on Python functions, pandas indexing, or file paths?

If needed, revisit the notebook, ask for hints, and then inspect the solutions file.


# Course structure

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A0 highlight;
```

In [ ]:
# Optional:
# bc.show_hint("q8")
# sol.show_solution("q8")
# sol.show_all()
